# 🏥 Hospital 30-Day Readmission Risk Predictor
**Author:** Shady Mohamed  
**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008) — 101,766 patient records  
**Goal:** Predict whether a diabetic patient will be readmitted within 30 days of discharge

---
### Problem Statement
Hospital readmissions within 30 days cost the US healthcare system ~$26 billion annually. Early identification of high-risk patients allows hospitals to intervene with targeted follow-up care, reducing costs and improving patient outcomes. This project builds an end-to-end ML pipeline to flag such patients at discharge.

### Pipeline Overview
1. Data loading & exploration (EDA)
2. Data cleaning & preprocessing
3. Feature engineering
4. Model training & comparison (5 models)
5. Hyperparameter tuning
6. SHAP explainability
7. Business impact analysis

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
%pip install matplotlib

import matplotlib.pyplot as plt

%pip install matplotlib

import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    f1_score
)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED']
print('All libraries loaded successfully!')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED']
print('All libraries loaded successfully!')
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve, AvgPrecisionScore, average_precision_score, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
import shap
import joblib
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, average_precision_score, f1_score

# Style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED']
print('All libraries loaded successfully!')

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
All libraries loaded successfully!
All libraries loaded successfully!


ImportError: cannot import name 'AvgPrecisionScore' from 'sklearn.metrics' (e:\Study\hospital_readmission_project\venv\Lib\site-packages\sklearn\metrics\__init__.py)

In [ ]:
# Download dataset from UCI repository
import urllib.request, zipfile, os

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00296/dataset_diabetes.zip'
if not os.path.exists('../data/diabetic_data.csv'):
    print('Downloading dataset...')
    urllib.request.urlretrieve(URL, '../data/diabetes.zip')
    with zipfile.ZipFile('../data/diabetes.zip', 'r') as z:
        z.extractall('../data/')
    print('Downloaded and extracted!')

df = pd.read_csv('../data/diabetic_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print(f'\nTarget distribution:')
print(df['readmitted'].value_counts())
print(f'\nMissing values (shown as ?):')
for col in df.columns:
    n = (df[col] == '?').sum()
    if n > 0:
        print(f'  {col}: {n:,} ({n/len(df)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA: Key Feature Distributions', fontsize=16, fontweight='bold', y=1.02)

# 1. Target distribution
target_counts = df['readmitted'].value_counts()
axes[0,0].bar(target_counts.index, target_counts.values, color=COLORS[:3])
axes[0,0].set_title('Readmission Distribution')
axes[0,0].set_xlabel('Readmitted')
axes[0,0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0,0].text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

# 2. Age distribution
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
age_counts = df['age'].value_counts().reindex(age_order)
axes[0,1].bar(range(len(age_order)), age_counts.values, color=COLORS[0])
axes[0,1].set_xticks(range(len(age_order)))
axes[0,1].set_xticklabels(age_order, rotation=45, ha='right', fontsize=8)
axes[0,1].set_title('Age Group Distribution')

# 3. Time in hospital
df['time_in_hospital'].hist(bins=14, ax=axes[0,2], color=COLORS[1], edgecolor='white')
axes[0,2].set_title('Time in Hospital (days)')
axes[0,2].set_xlabel('Days')

# 4. Number of medications
df['num_medications'].hist(bins=30, ax=axes[1,0], color=COLORS[2], edgecolor='white')
axes[1,0].set_title('Number of Medications')

# 5. Number of diagnoses
df['number_diagnoses'].hist(bins=16, ax=axes[1,1], color=COLORS[3], edgecolor='white')
axes[1,1].set_title('Number of Diagnoses')

# 6. Readmission by age
early_by_age = df[df['readmitted']=='<30'].groupby('age').size().reindex(age_order)
total_by_age = df.groupby('age').size().reindex(age_order)
rate_by_age = (early_by_age / total_by_age * 100).fillna(0)
axes[1,2].bar(range(len(age_order)), rate_by_age.values, color=COLORS[4])
axes[1,2].set_xticks(range(len(age_order)))
axes[1,2].set_xticklabels(age_order, rotation=45, ha='right', fontsize=8)
axes[1,2].set_title('Early Readmission Rate by Age (%)')

plt.tight_layout()
plt.savefig('../reports/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Cleaning & Preprocessing

In [ ]:
df_clean = df.copy()

# Replace '?' with NaN
df_clean.replace('?', np.nan, inplace=True)

# Drop high-missingness columns (>40% missing)
high_missing = ['weight', 'payer_code', 'medical_specialty']
df_clean.drop(columns=high_missing, inplace=True)
print(f'Dropped high-missing columns: {high_missing}')

# Drop duplicates (keep first encounter per patient)
df_clean.drop_duplicates(subset='patient_nbr', keep='first', inplace=True)
print(f'After dedup: {df_clean.shape[0]:,} rows')

# Drop rows where discharge = expired or hospice
df_clean = df_clean[~df_clean['discharge_disposition_id'].isin([11, 13, 14, 19, 20, 21])]
print(f'After removing discharged-to-hospice: {df_clean.shape[0]:,} rows')

# Binary target: 1 = readmitted within 30 days, 0 = not
df_clean['target'] = (df_clean['readmitted'] == '<30').astype(int)
print(f"\nTarget balance: {df_clean['target'].value_counts().to_dict()}")
print(f"Early readmission rate: {df_clean['target'].mean()*100:.1f}%")

## 4. Feature Engineering

In [ ]:
df_feat = df_clean.copy()

# --- Engineered features ---

# 1. Medication change flag
med_cols = ['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
            'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
            'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
            'examide','citoglipton','insulin','glyburide-metformin','glipizide-metformin',
            'glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone']
df_feat['n_meds_changed'] = (df_feat[med_cols].isin(['Up','Down'])).sum(axis=1)
df_feat['any_med_change'] = (df_feat['n_meds_changed'] > 0).astype(int)

# 2. Service utilization score
df_feat['utilization_score'] = (df_feat['number_outpatient'].astype(float) +
                                 df_feat['number_emergency'].astype(float) +
                                 df_feat['number_inpatient'].astype(float))

# 3. Age as numeric midpoint
age_map = {'[0-10)':5,'[10-20)':15,'[20-30)':25,'[30-40)':35,'[40-50)':45,
           '[50-60)':55,'[60-70)':65,'[70-80)':75,'[80-90)':85,'[90-100)':95}
df_feat['age_num'] = df_feat['age'].map(age_map)

# 4. High HbA1c flag (A1Cresult > 8 = poor glycemic control)
df_feat['high_a1c'] = df_feat['A1Cresult'].isin(['>8', '>7']).astype(int)

# 5. Comorbidity score (simplified Charlson proxy)
df_feat['comorbidity_score'] = (df_feat['number_diagnoses'].astype(float) +
                                 df_feat['time_in_hospital'].astype(float) / 3)

# 6. Prior visits flag
df_feat['had_prior_inpatient'] = (df_feat['number_inpatient'].astype(float) > 0).astype(int)
df_feat['had_prior_emergency'] = (df_feat['number_emergency'].astype(float) > 0).astype(int)

print('Engineered features added:')
new_feats = ['n_meds_changed','any_med_change','utilization_score','age_num',
             'high_a1c','comorbidity_score','had_prior_inpatient','had_prior_emergency']
for f in new_feats:
    print(f'  {f}: {df_feat[f].dtype}')

In [ ]:
# Select final feature set
NUMERIC_FEATURES = [
    'age_num', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient',
    'number_diagnoses', 'n_meds_changed', 'utilization_score', 'comorbidity_score'
]

BINARY_FEATURES = [
    'any_med_change', 'high_a1c', 'had_prior_inpatient', 'had_prior_emergency',
    'change', 'diabetesMed'
]

CATEGORICAL_FEATURES = ['race', 'gender', 'admission_type_id',
                         'discharge_disposition_id', 'admission_source_id']

# Encode binary cols
df_feat['change'] = (df_feat['change'] == 'Ch').astype(int)
df_feat['diabetesMed'] = (df_feat['diabetesMed'] == 'Yes').astype(int)
df_feat['gender'] = (df_feat['gender'] == 'Male').astype(int)

# Encode categoricals
le = LabelEncoder()
for col in ['race', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']:
    df_feat[col] = df_feat[col].fillna('Unknown')
    df_feat[col] = le.fit_transform(df_feat[col].astype(str))

ALL_FEATURES = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES
X = df_feat[ALL_FEATURES].copy()
y = df_feat['target'].copy()

# Handle any remaining missing values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=ALL_FEATURES)

print(f'Feature matrix: {X_imputed.shape}')
print(f'Class balance: {y.value_counts().to_dict()}')

## 5. Train/Test Split & Class Imbalance Handling (SMOTE)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_sc, y_train)

print(f'Train size: {X_train_sc.shape[0]:,} → After SMOTE: {X_train_bal.shape[0]:,}')
print(f'Test size:  {X_test_sc.shape[0]:,}')
print(f'Balanced class distribution: {pd.Series(y_train_bal).value_counts().to_dict()}')

## 6. Model Training & Comparison (5 Models)

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, use_label_encoder=False,
                                         eval_metric='logloss', random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=42)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in MODELS.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train_bal, y_train_bal)
    y_pred  = model.predict(X_test_sc)
    y_prob  = model.predict_proba(X_test_sc)[:,1] if hasattr(model, 'predict_proba') else None
    
    auc  = roc_auc_score(y_test, y_prob) if y_prob is not None else 0
    f1   = f1_score(y_test, y_pred)
    apr  = average_precision_score(y_test, y_prob) if y_prob is not None else 0
    cv_f1 = cross_val_score(model, X_train_bal, y_train_bal, cv=cv,
                             scoring='f1', n_jobs=-1).mean()
    
    results[name] = {'AUC-ROC': auc, 'F1': f1, 'Avg Precision': apr, 'CV-F1': cv_f1,
                     'model': model, 'y_prob': y_prob, 'y_pred': y_pred}
    print(f'AUC={auc:.3f} | F1={f1:.3f} | CV-F1={cv_f1:.3f}')

print('\nAll models trained!')

In [ ]:
# Comparison chart
metrics_df = pd.DataFrame({k: {m: results[k][m] for m in ['AUC-ROC','F1','Avg Precision','CV-F1']}
                           for k in results}).T.sort_values('AUC-ROC', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Comparison', fontsize=15, fontweight='bold')

# Grouped bar chart
x = np.arange(len(metrics_df))
width = 0.2
for i, metric in enumerate(['AUC-ROC','F1','Avg Precision','CV-F1']):
    axes[0].bar(x + i*width, metrics_df[metric], width, label=metric, color=COLORS[i])
axes[0].set_xticks(x + width*1.5)
axes[0].set_xticklabels(metrics_df.index, rotation=20, ha='right')
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].set_title('Performance Metrics by Model')

# ROC curves
axes[1].plot([0,1],[0,1],'k--', label='Random', alpha=0.5)
for i, (name, res) in enumerate(results.items()):
    if res['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
        axes[1].plot(fpr, tpr, color=COLORS[i],
                     label=f"{name} (AUC={res['AUC-ROC']:.3f})", linewidth=2)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(metrics_df[['AUC-ROC','F1','Avg Precision','CV-F1']].to_string())

## 7. Hyperparameter Tuning (Best Model — XGBoost)

In [ ]:
param_dist = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6, 8],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.3],
    'reg_alpha':        [0, 0.01, 0.1],
    'reg_lambda':       [1, 1.5, 2]
}

base_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                          random_state=42, n_jobs=-1)

rs = RandomizedSearchCV(
    base_xgb, param_dist, n_iter=40,
    scoring='roc_auc', cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1, random_state=42, verbose=1
)
rs.fit(X_train_bal, y_train_bal)

best_xgb = rs.best_estimator_
y_pred_best = best_xgb.predict(X_test_sc)
y_prob_best = best_xgb.predict_proba(X_test_sc)[:,1]

print(f'Best params: {rs.best_params_}')
print(f'\nTuned XGBoost Performance:')
print(f'  AUC-ROC: {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'  F1:      {f1_score(y_test, y_pred_best):.4f}')
print(f'  Avg Prec:{average_precision_score(y_test, y_prob_best):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Not Readmitted','Readmitted <30d']))

## 8. SHAP Explainability

In [ ]:
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test_sc[:2000])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Explainability (SHAP)', fontsize=15, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(shap_values, pd.DataFrame(X_test_sc[:2000], columns=ALL_FEATURES),
                  plot_type='bar', show=False)
axes[0].set_title('Top Feature Importances (SHAP)')

plt.sca(axes[1])
shap.summary_plot(shap_values, pd.DataFrame(X_test_sc[:2000], columns=ALL_FEATURES), show=False)
axes[1].set_title('SHAP Value Distribution')

plt.tight_layout()
plt.savefig('../reports/shap_explainability.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP analysis complete — see which features drive readmission risk most!')

## 9. Business Impact Analysis

In [ ]:
# Threshold optimization for clinical use case
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []
for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    tn = ((y_pred_t==0) & (y_test==0)).sum()
    fp = ((y_pred_t==1) & (y_test==0)).sum()
    fn = ((y_pred_t==0) & (y_test==1)).sum()
    tp = ((y_pred_t==1) & (y_test==1)).sum()
    # Assume: cost of missed readmission = $15,000, cost of unnecessary intervention = $500
    cost = fn * 15000 + fp * 500
    threshold_results.append({'threshold': t, 'tp': tp, 'fp': fp,
                               'fn': fn, 'tn': tn, 'cost': cost,
                               'recall': tp/(tp+fn) if (tp+fn)>0 else 0,
                               'precision': tp/(tp+fp) if (tp+fp)>0 else 0})

thr_df = pd.DataFrame(threshold_results)
optimal_idx = thr_df['cost'].idxmin()
optimal = thr_df.iloc[optimal_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thr_df['threshold'], thr_df['cost']/1e6, color=COLORS[2], linewidth=2)
axes[0].axvline(x=optimal['threshold'], color=COLORS[0], linestyle='--',
                label=f"Optimal threshold: {optimal['threshold']:.2f}")
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Estimated Cost ($M)')
axes[0].set_title('Cost Optimization by Threshold')
axes[0].legend()

axes[1].plot(thr_df['threshold'], thr_df['recall'], label='Recall', color=COLORS[0], linewidth=2)
axes[1].plot(thr_df['threshold'], thr_df['precision'], label='Precision', color=COLORS[1], linewidth=2)
axes[1].axvline(x=optimal['threshold'], color=COLORS[2], linestyle='--', label='Optimal')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision-Recall Tradeoff')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/business_impact.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n=== BUSINESS IMPACT SUMMARY ===')
print(f'Optimal threshold: {optimal["threshold"]:.2f}')
print(f'At this threshold:')
print(f'  True positives (caught):  {int(optimal["tp"]):,} high-risk patients flagged')
print(f'  False negatives (missed): {int(optimal["fn"]):,} readmissions not caught')
print(f'  Recall:    {optimal["recall"]*100:.1f}%')
print(f'  Precision: {optimal["precision"]*100:.1f}%')
print(f'  Estimated cost: ${optimal["cost"]/1e6:.2f}M')
baseline_cost = (y_test==1).sum() * 15000
print(f'  Baseline cost (no model): ${baseline_cost/1e6:.2f}M')
print(f'  Estimated savings: ${(baseline_cost - optimal["cost"])/1e6:.2f}M')

## 10. Save Model & Artifacts

In [ ]:
joblib.dump(best_xgb, '../models/readmission_xgb_tuned.pkl')
joblib.dump(scaler,   '../models/scaler.pkl')
joblib.dump(imputer,  '../models/imputer.pkl')

model_summary = {
    'model': 'XGBoost (Tuned)',
    'features': ALL_FEATURES,
    'best_params': rs.best_params_,
    'auc_roc': roc_auc_score(y_test, y_prob_best),
    'f1_score': f1_score(y_test, y_pred_best),
    'optimal_threshold': float(optimal['threshold']),
    'training_samples': len(X_train_bal),
    'test_samples': len(X_test)
}
import json
with open('../models/model_summary.json', 'w') as f:
    json.dump(model_summary, f, indent=2)

print('Model and artifacts saved!')
print('\n=== FINAL PROJECT SUMMARY ===')
print(f'Dataset: {df_clean.shape[0]:,} patient encounters')
print(f'Features used: {len(ALL_FEATURES)} (incl. 8 engineered)')
print(f'Best model: Tuned XGBoost')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_best):.4f}')
print('Reports saved to: ../reports/')
print('Models saved to:  ../models/')